# 完整的動手指南：可在 Google Colab GPU 上訓練你的神經網路模型

這份期中報告以 **Python、PyTorch 與 Fashion-MNIST** 示範完整的神經網路實驗。
請由上往下依序執行，每個程式格都已包含在本筆記本，不必先複製 GitHub 專案。

完成後，你將能啟用並確認 GPU、切分資料、建立 CNN、訓練與挑選最佳模型，
最後評估測試集、閱讀圖表、保存權重並載入模型進行推論。
本筆記本沒有預填執行成果；準確率與耗時由你實際執行後產生。


## 1. 開啟 Colab 並啟用 GPU

1. 在 [Google Colab](https://colab.research.google.com/) 選擇「檔案 → 上傳筆記本」，
   上傳 `colab_gpu_training.ipynb`，或直接從自己的 GitHub repository 開啟此檔。
2. 選擇「執行階段 → 變更執行階段類型」，將硬體加速器設為 **GPU**，再連線。
   介面名稱可能隨語言或版本略有不同。
3. 從第一個程式格開始依序執行；變更硬體後要重新執行全部程式格。

Colab 的 GPU 型號、可用額度與連線時間會變動，免費資源不保證供應。
本教學預設 `REQUIRE_GPU = True`：未偵測到 CUDA 時會停止並給出提示。
若要先在 CPU 檢查流程，可在下方參數格明確改成 `False`，並在報告標示 **CPU**；
CPU 執行不能當作 Colab GPU 實驗結果。
請參考 [Colab 官方資源限制說明](https://research.google.com/colaboratory/faq.html#resource-limits)。

Colab 預裝的 PyTorch 與 torchvision 應保留。下格只補裝缺少的 NumPy 或 Matplotlib，
不重裝 CUDA、torch 或 torchvision；若這兩個主要套件缺少，請先確認你使用的是標準 Colab Python 執行階段。


In [ ]:
import importlib.util
import subprocess
import sys

missing = [name for name in ("numpy", "matplotlib")
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
for name in ("torch", "torchvision"):
    if importlib.util.find_spec(name) is None:
        raise ModuleNotFoundError(
            f"缺少 {name}。請使用標準 Colab Python 執行階段；"
            "本機環境請依 README 安裝相容的 PyTorch / torchvision。"
        )
print("套件檢查完成；保留目前的 PyTorch / torchvision。")


In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
import csv
import json
import math
import platform
import random
import time

import numpy as np
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt


@dataclass
class Config:
    epochs: int = 5
    batch_size: int = 128
    learning_rate: float = 0.001
    seed: int = 42
    val_size: int = 6000
    num_workers: int = 0
    quick: bool = False
    require_gpu: bool = False

    def __post_init__(self):
        if self.epochs < 1 or self.batch_size < 1:
            raise ValueError("epochs and batch_size must be positive")
        if not math.isfinite(self.learning_rate) or self.learning_rate <= 0:
            raise ValueError("learning_rate must be finite and positive")
        if self.val_size < 1 or self.num_workers < 0:
            raise ValueError("val_size must be positive; num_workers cannot be negative")
        if self.quick:
            self.epochs = 1


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def get_device(require_gpu=False):
    if torch.cuda.is_available():
        return torch.device("cuda")
    if require_gpu:
        raise RuntimeError(
            "CUDA GPU unavailable. In Colab choose Runtime > Change runtime type > GPU, "
            "then reconnect. To intentionally use CPU set REQUIRE_GPU=False "
            "or omit --require-gpu. GPU availability is not guaranteed."
        )
    return torch.device("cpu")


def environment_info(device):
    return {
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": str(torch.__version__),
        "torchvision": str(torchvision.__version__),
        "numpy": str(np.__version__),
        "device": str(device),
        "cuda_available": torch.cuda.is_available(),
        "cuda_runtime": torch.version.cuda,
        "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "cpu_threads": torch.get_num_threads(),
    }


## 2. 資料與可重現的實驗設計

Fashion-MNIST 是 28 × 28 灰階服飾影像，共十類。原始 60,000 張訓練影像固定以
seed 42 切為 **54,000 張訓練、6,000 張驗證**；官方 **10,000 張測試影像**留到模型挑選完才評估。
訓練集更新權重，驗證集用來挑選 checkpoint，測試集用來估計最後模型的表現。

每張影像先轉為 `[1, 28, 28]` 浮點張量，再以 `(像素值 − 0.5) / 0.5` 正規化，
將 0～1 轉為約 −1～1。這是預先固定的常數，不從測試集估計統計量。
固定種子有助於重現切分與訓練；不同硬體、套件版本仍可能產生數值差異。

下格定義共用資料函式，尚未開始訓練。類別保留英文名稱，避免圖表中文字型缺失。


In [ ]:
CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]


class FashionMNIST(datasets.FashionMNIST):
    # Official dataset repository over HTTPS; torchvision still checks the MD5 hashes.
    mirrors = [
        "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/",
    ]


def split_indices(size, val_size, seed):
    if not 0 < val_size < size:
        raise ValueError("val_size must leave at least one training sample")
    order = torch.randperm(size, generator=torch.Generator().manual_seed(seed)).tolist()
    return order[val_size:], order[:val_size]


def make_loaders(config, data_dir="data"):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    full_train = FashionMNIST(data_dir, train=True, download=True, transform=transform)
    test_set = FashionMNIST(data_dir, train=False, download=True, transform=transform)
    train_ids, val_ids = split_indices(len(full_train), config.val_size, config.seed)
    train_set = Subset(full_train, train_ids)
    val_set = Subset(full_train, val_ids)
    if config.quick:
        # Only for smoke testing, never report these as full-dataset results.
        train_set = Subset(train_set, range(min(2048, len(train_set))))
        val_set = Subset(val_set, range(min(512, len(val_set))))
        test_set = Subset(test_set, range(min(512, len(test_set))))
    options = {"batch_size": config.batch_size, "num_workers": config.num_workers,
               "pin_memory": torch.cuda.is_available()}
    train_loader = DataLoader(train_set, shuffle=True,
                              generator=torch.Generator().manual_seed(config.seed), **options)
    return (train_loader, DataLoader(val_set, shuffle=False, **options),
            DataLoader(test_set, shuffle=False, **options))


## 3. 設定參數與保存位置

預設為 **5 epochs、batch size 128、Adam 學習率 0.001**。
`QUICK = False` 使用完整資料；改成 `True` 時，訓練／驗證／測試最多只用 2,048／512／512 張，
且 `Config` 會把 epochs 設為 1 來檢查流程，結果不能代表完整實驗。
想比較參數時，請保持資料切分與測試流程一致，以驗證集決定設定，最後才看測試成績。

Colab 暫存磁碟可能隨執行階段結束而清除。可把 `USE_GOOGLE_DRIVE` 改成 `True`，
依畫面授權掛載自己的 Drive，讓結果直接寫入 `MyDrive/colab_gpu_training`；資料下載仍放在本機暫存。
每次執行本格會建立新的時間戳記資料夾，避免沿用先前的 checkpoint。
若要重跑完整實驗，請從本格開始依序重新執行。


In [ ]:
REQUIRE_GPU = True
QUICK = False
USE_GOOGLE_DRIVE = False
EPOCHS = 5

CONFIG = Config(
    epochs=EPOCHS,
    batch_size=128,
    learning_rate=0.001,
    seed=42,
    val_size=6000,
    num_workers=0,
    quick=QUICK,
    require_gpu=REQUIRE_GPU,
)
DATA_DIR = Path("data")
OUTPUT_ROOT = Path("outputs")
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/colab_gpu_training")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime("colab_%Y%m%dT%H%M%S_%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("參數：", asdict(CONFIG))
print("成果資料夾：", RUN_DIR.resolve())


## 4. 確認真正使用的裝置

`torch.cuda.is_available()` 是判斷 PyTorch 能否使用 CUDA 的依據；只選取 Colab GPU 選項並不足夠。
下格會印出實際裝置、GPU 名稱與套件版本。報告請保留這些環境資訊。
稍後模型會用 `model.to(device)` 移到此裝置，每個 batch 的影像與標籤也必須移到同一裝置。

如果這格因缺少 GPU 停止：確認加速器設定並重新連線，或等可用資源恢復後再試；
要明確採用 CPU 備用模式，回到參數格設定 `REQUIRE_GPU = False` 再依序重跑。


In [ ]:
seed_everything(CONFIG.seed)
device = get_device(require_gpu=CONFIG.require_gpu)
print(json.dumps(environment_info(device), ensure_ascii=False, indent=2))
print("本次模式：", "快速流程檢查" if CONFIG.quick else "完整資料實驗")


## 5. 下載、切分並預覽訓練影像

第一次執行會下載 Fashion-MNIST；後續會重用 `DATA_DIR` 的檔案。
DataLoader 將資料分成 batch，訓練資料會打亂，驗證與測試資料不打亂。
下方只預覽訓練集，並用 `image * 0.5 + 0.5` 還原顯示亮度。
請確認影像、類別與切分數量合理；快速模式會顯示縮小後的實際數量。


In [ ]:
train_loader, val_loader, test_loader = make_loaders(CONFIG, data_dir=DATA_DIR)
split_sizes = {
    "train": len(train_loader.dataset),
    "validation": len(val_loader.dataset),
    "test": len(test_loader.dataset),
}
print("資料切分：", split_sizes)

fig, axes = plt.subplots(3, 4, figsize=(9, 7))
for index, axis in enumerate(axes.flat):
    image, label = train_loader.dataset[index]
    axis.imshow((image.squeeze(0) * 0.5 + 0.5).numpy(), cmap="gray", vmin=0, vmax=1)
    axis.set_title(CLASS_NAMES[int(label)])
    axis.axis("off")
fig.suptitle("Training samples")
fig.tight_layout()
plt.show()
plt.close(fig)


## 6. 建立卷積神經網路（CNN）

卷積層學習局部影像特徵，池化縮小空間大小，ReLU 提供非線性，Dropout 在訓練時減少過度依賴少數特徵。
最後輸出十個 **logits**。`CrossEntropyLoss` 直接接受 logits，所以模型最後不加 Softmax。

| 階段 | 單張影像的形狀 |
| --- | --- |
| 輸入 | 1 × 28 × 28 |
| Conv 1→16（3×3、padding 1）＋ReLU＋2×2 MaxPool | 16 × 14 × 14 |
| Conv 16→32（3×3、padding 1）＋ReLU＋2×2 MaxPool | 32 × 7 × 7 |
| 展平 | 1,568 |
| Linear＋ReLU＋Dropout | 128 |
| Linear | 10 |

模型共有 **206,922 個可訓練參數**；下格會計算確認。`[batch, 1, 28, 28]` 輸入應得到 `[batch, 10]`。


In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10),
        )

    def forward(self, images):
        # Return logits: CrossEntropyLoss applies log-softmax internally.
        return self.classifier(self.features(images))


In [ ]:
model = FashionCNN().to(device)
print(model)
parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"可訓練參數：{parameter_count:,}")
model.eval()
with torch.no_grad():
    example_input = torch.zeros(2, 1, 28, 28, device=device)
    example_output = model(example_input)
print("輸入形狀：", tuple(example_input.shape), "輸出形狀：", tuple(example_output.shape))


## 7. 訓練與驗證

一個 epoch 代表走過一次訓練資料。每個 batch 依序執行：
**將張量移至裝置 → 清除梯度 → 前向傳播 → 計算 loss → 反向傳播 → Adam 更新權重**。
`model.train()` 啟用訓練行為；驗證時用 `model.eval()` 與 `torch.inference_mode()`，不更新參數。

每輪分別記錄訓練與驗證的 loss、accuracy。平均 loss 依樣本數加權，避免最後一個較小 batch 被算得太重。
`fit` 只依 **最低驗證 loss** 保存 `best_model.pt`，不查看測試成績來挑選模型。
訓練函式的原始碼放在下格，可直接看到 `images.to(device)` 與 `labels.to(device)`。


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss, correct, count = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        count += labels.size(0)
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
    if count == 0:
        raise ValueError("Training loader must not be empty")
    return total_loss / count, correct / count


@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    total_loss, count = 0.0, 0
    labels_all, predictions_all = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        total_loss += loss_fn(logits, labels).item()
        count += labels.size(0)
        labels_all.append(labels.cpu())
        predictions_all.append(logits.argmax(1).cpu())
    if count == 0:
        raise ValueError("Evaluation loader must not be empty")
    y_true, y_pred = torch.cat(labels_all), torch.cat(predictions_all)
    return {"loss": total_loss / count,
            "accuracy": (y_true == y_pred).sum().item() / count,
            "y_true": y_true, "y_pred": y_pred}


def fit(model, train_loader, val_loader, config, device, run_dir):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    history, best_loss, best_epoch = [], float("inf"), 0
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    started = time.perf_counter()
    for epoch in range(1, config.epochs + 1):
        epoch_started = time.perf_counter()
        train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, device)
        validation = evaluate(model, val_loader, device)
        if not math.isfinite(train_loss) or not math.isfinite(validation["loss"]):
            raise RuntimeError("Non-finite loss. Check inputs and reduce the learning rate.")
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        row = {"epoch": epoch, "train_loss": train_loss, "train_accuracy": train_accuracy,
               "val_loss": validation["loss"], "val_accuracy": validation["accuracy"],
               "epoch_seconds": time.perf_counter() - epoch_started}
        history.append(row)
        if validation["loss"] < best_loss:
            best_loss, best_epoch = validation["loss"], epoch
            torch.save({"model_state_dict": model.state_dict(), "epoch": epoch,
                        "val_loss": best_loss, "config": asdict(config)},
                       run_dir / "best_model.pt")
        # Flush the learning history after every epoch so a disconnect loses less work.
        with (run_dir / "history.csv").open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(row))
            writer.writeheader()
            writer.writerows(history)
        print(f"Epoch {epoch:02d}/{config.epochs} | loss {train_loss:.4f} | "
              f"acc {train_accuracy:.2%} | val_loss {validation['loss']:.4f} | "
              f"val_acc {validation['accuracy']:.2%}", flush=True)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    return history, best_epoch, time.perf_counter() - started


In [ ]:
history, best_epoch, train_seconds = fit(
    model, train_loader, val_loader, CONFIG, device, RUN_DIR
)
print(f"最低驗證 loss 位於第 {best_epoch} 輪；訓練耗時 {train_seconds:.2f} 秒。")


## 8. 還原最佳模型

最後一輪不一定是驗證表現最佳的一輪。評估測試集前，先載入同一實驗資料夾中保存的最佳權重。
`map_location=device` 讓權重載入目前裝置；`weights_only=True` 用於本專案儲存的基本資料與張量 checkpoint。
此處只載入自己剛產生的檔案。


In [ ]:
checkpoint = torch.load(RUN_DIR / "best_model.pt", map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()
print("已還原第", checkpoint["epoch"], "輪，驗證 loss =", checkpoint["val_loss"])


## 9. 最後一次測試與圖表分析

下方 `save_artifacts` **對測試集評估一次**，再以同一份預測製作混淆矩陣與預測圖。
Accuracy 是分類正確的比例；loss 也反映預測信心，兩者需一起閱讀。
混淆矩陣的 **列是真實類別、欄是預測類別**，對角線以外的數量能指出容易混淆的服飾。

訓練 loss 持續下降但驗證 loss 上升，可能出現過擬合。請據驗證曲線討論原因，
不要看到測試成績後反覆修改超參數、再挑最高測試分數。
以下所有數字與圖表都由本次實際執行產生；快速模式的測試樣本也較少。

成果包含 `history.csv`、`metrics.json`、`environment.json`、`config.json`、`best_model.pt`，
以及訓練曲線、混淆矩陣、預測圖；有錯誤預測時另存 `misclassified.png`。


In [ ]:
def confusion_counts(y_true, y_pred, num_classes=10):
    counts = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(counts, (np.asarray(y_true), np.asarray(y_pred)), 1)
    return counts


def plot_history(history, path):
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for prefix, label in [("train", "Training"), ("val", "Validation")]:
        axes[0].plot(epochs, [row[prefix + "_loss"] for row in history], "o-", label=label)
        axes[1].plot(epochs, [row[prefix + "_accuracy"] for row in history], "o-", label=label)
    for axis, title in zip(axes, ["Cross-entropy loss", "Accuracy"]):
        axis.set(xlabel="Epoch", title=title, xticks=epochs)
        axis.grid(alpha=0.2)
        axis.legend()
    axes[1].set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_confusion(counts, path):
    fig, axis = plt.subplots(figsize=(10, 9))
    chart = axis.imshow(counts, cmap="Blues")
    fig.colorbar(chart, ax=axis, fraction=0.046, pad=0.04)
    axis.set(xticks=range(10), yticks=range(10), xticklabels=CLASS_NAMES,
             yticklabels=CLASS_NAMES, xlabel="Predicted class", ylabel="True class",
             title="Test confusion matrix (counts)")
    plt.setp(axis.get_xticklabels(), rotation=45, ha="right")
    for row in range(10):
        for col in range(10):
            axis.text(col, row, str(counts[row, col]), ha="center", va="center", fontsize=8,
                      color="white" if counts[row, col] > counts.max() / 2 else "black")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_predictions(dataset, y_true, y_pred, path, errors_only=False):
    indices = torch.where(y_true != y_pred)[0].tolist() if errors_only else list(range(len(y_true)))
    indices = indices[:12]
    if not indices:
        return False
    rows = math.ceil(len(indices) / 4)
    fig, axes = plt.subplots(rows, 4, figsize=(12, rows * 3), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, index in zip(axes.flat, indices):
        image, _ = dataset[index]
        axis.imshow((image.squeeze().numpy() * 0.5 + 0.5).clip(0, 1), cmap="gray")
        true_id, predicted_id = y_true[index].item(), y_pred[index].item()
        axis.set_title(f"True: {CLASS_NAMES[true_id]}\nPred: {CLASS_NAMES[predicted_id]}",
                       color="green" if true_id == predicted_id else "crimson", fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return True


def save_artifacts(model, test_loader, history, config, device, run_dir,
                   best_epoch, train_seconds, split_sizes):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    result = evaluate(model, test_loader, device)
    counts = confusion_counts(result["y_true"], result["y_pred"])
    per_class = []
    for index, name in enumerate(CLASS_NAMES):
        true_positive = int(counts[index, index])
        predicted = int(counts[:, index].sum())
        support = int(counts[index].sum())
        precision = true_positive / predicted if predicted else 0.0
        recall = true_positive / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class.append({"class_id": index, "class_name": name, "precision": precision,
                          "recall": recall, "f1": f1, "support": support})
    metrics = {"test_loss": result["loss"], "test_accuracy": result["accuracy"],
               "best_epoch": best_epoch,
               "best_val_loss": min(row["val_loss"] for row in history),
               "train_seconds": train_seconds,
               "parameter_count": sum(p.numel() for p in model.parameters()),
               "split_sizes": split_sizes, "quick_mode": config.quick,
               "per_class": per_class,
               "macro_f1": sum(row["f1"] for row in per_class) / len(per_class)}
    for name, payload in [("config.json", asdict(config)),
                          ("environment.json", environment_info(device)), ("metrics.json", metrics)]:
        (run_dir / name).write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    plot_history(history, run_dir / "training_curves.png")
    plot_confusion(counts, run_dir / "confusion_matrix.png")
    plot_predictions(test_loader.dataset, result["y_true"], result["y_pred"], run_dir / "predictions.png")
    plot_predictions(test_loader.dataset, result["y_true"], result["y_pred"],
                     run_dir / "misclassified.png", errors_only=True)
    print(f"Best epoch: {best_epoch} | Test accuracy: {result['accuracy']:.2%}", flush=True)
    return metrics


In [ ]:
metrics = save_artifacts(
    model, test_loader, history, CONFIG, device, RUN_DIR,
    best_epoch, train_seconds, split_sizes,
)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
try:
    from IPython.display import Image, display
except ImportError:
    Image = None
for filename in ("training_curves.png", "confusion_matrix.png", "predictions.png", "misclassified.png"):
    image_path = RUN_DIR / filename
    if image_path.exists():
        if Image is not None:
            display(Image(filename=str(image_path), width=900))
        else:
            print("圖表：", image_path.resolve())
print("成果檔案：", [path.name for path in sorted(RUN_DIR.iterdir())])


## 10. 載入權重進行單張推論

新建相同結構的模型並載入權重，就能恢復已訓練的模型。
下格從已完成最終評估的測試資料中取一張影像，示範 `unsqueeze(0)` 加上 batch 維度，
然後以 `argmax` 取最高分的類別。這只是單張推論示範，不重新計算測試指標。
預測錯誤也應如實保留並討論；單張成功案例不能代表整體準確率。


In [ ]:
reloaded_model = FashionCNN().to(device)
saved_checkpoint = torch.load(RUN_DIR / "best_model.pt", map_location=device, weights_only=True)
reloaded_model.load_state_dict(saved_checkpoint["model_state_dict"])
reloaded_model.eval()
sample_image, sample_label = test_loader.dataset[0]
with torch.no_grad():
    sample_logits = reloaded_model(sample_image.unsqueeze(0).to(device))
    sample_prediction = int(sample_logits.argmax(dim=1).item())
print("真實類別：", CLASS_NAMES[int(sample_label)])
print("預測類別：", CLASS_NAMES[sample_prediction])


## 11. 保存與下載實驗成果

先確認上方列出的檔案已產生。若使用暫存磁碟，離開前可把 `DOWNLOAD_RESULTS` 改成 `True`，
將本次資料夾壓縮後下載到自己的電腦。Drive 使用者也可直接在 Drive 中取回成果。
提交報告時請記錄 seed、實際切分大小、epochs、最佳輪次、測試指標、耗時與 GPU 型號，
並附圖說明觀察到的現象；本機 CPU 驗證與 Colab GPU 實驗需分開標示。

你也可以在 Colab 選擇「檔案 → 下載 → 下載 .ipynb」保存執行輸出，再上傳到自己的 GitHub。
分享前檢查筆記本內是否含有私人 Drive 路徑或其他個人資訊。


In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files
    archive_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
    files.download(archive_path)
else:
    print("成果保存在：", RUN_DIR.resolve())


## 參考資料與延伸練習

- [Google Colab 官方 FAQ：GPU 與資源限制](https://research.google.com/colaboratory/faq.html)
- [PyTorch 官方 Quickstart：資料、模型、訓練與保存](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html)
- [torchvision 官方 FashionMNIST 文件](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html)
- [PyTorch 官方 torch.load 文件](https://docs.pytorch.org/docs/stable/generated/torch.load.html)
- [Fashion-MNIST 資料集與類別說明](https://github.com/zalandoresearch/fashion-mnist)

延伸時每次只改一項設定，例如學習率、Dropout 或 epoch 數，用驗證集比較並記錄理由。
完整報告應區分「實際觀察」、「可能原因」與「未來待驗證的假設」。
